# Imports

In [ ]:
from Functions.Fcts_Filtering import plot_random_organoids, filter_organoids_by, filter_rows_by_percentile_bounds
from Functions.Fcts_Base import save_after_filtering, load_adata
import pandas as pd


%load_ext autoreload
%autoreload 2

# User Input

In [ ]:
load_dir = "PATH TO ANNDATA OBJECT FROM 1_FeatureExtraction" # Path to .h5ad file containing the AnnData object.
pyramid_lvl_visualization = 1 # Used for visualization of images in plots. 0: full resolution, 1: downsampled by factor 2, etc.
channel_visualization = 0 # Channel used for visualization of images in plots.

# Load

In [ ]:
ad_raw = load_adata(load_dir, load_zarrs = True)

df_raw = pd.concat([ad_raw.to_df(), ad_raw.obs.astype(str)], axis = 1)

# Filter
A variety of filter functions are examplified below. They filter objects based on a specified numerical feature range and visualize the removed objects. To not overwrite an existing DataFrame, save the filtered DataFrame with a new name.    

### Cut Objects
Filters objects which are not completely imaged based on the fraction of the longest straight edge in the image. 

In [ ]:
feat = "StraightEdge_longest" # Name of the numerical feature to filter.
lower_threshold = 10 # Lower boundary. Every object below this threshold in the feature is removed.
upper_threshold = 50 # Upper boundary. Every object above this threshold in the feature is removed.

df_filt = filter_organoids_by(ad_raw,
                    df = df_raw,
                    feature = feat, 
                    values = (lower_threshold, upper_threshold),
                    channel = channel_visualization,
                    pyramid_level = pyramid_lvl_visualization
                    )

### Low DAPI brightness

In [ ]:
feat = "R0__DAPI_mean" # Name of the numerical feature to filter.
lower_threshold = 700 # Lower boundary
upper_threshold = 3000 # Upper boundary

df_filt = filter_organoids_by(ad_raw,
                    df = df_filt,
                    feature = feat, 
                    values = (lower_threshold, upper_threshold),
                    channel = channel_visualization,
                    pyramid_level = pyramid_lvl_visualization
                    )

### Low size

In [ ]:
feat = "area" # Name of the numerical feature to filter.
lower_threshold = 5 # Lower boundary
upper_threshold = 999999 # Upper boundary

df_filt = filter_organoids_by(ad_raw,
                    df = df_filt,
                    feature = feat, 
                    values = (lower_threshold, upper_threshold),
                    channel = channel_visualization,
                    pyramid_level = pyramid_lvl_visualization
                    )

### Percentile Filter

In [ ]:
features = ["area", "perimeter", "R0__DAPI_mean"]
lower_q = 0.01
upper_q = 0.99

df_filt = filter_rows_by_percentile_bounds(
    df_filt,
    features = features,
    lower_q = lower_q,
    upper_q = upper_q,
)

### Sample of remaining objects

In [ ]:
feature = "area" #Feature to display in title below unique ID

fig = plot_random_organoids(ad_raw,
                            df_raw = df_raw,
                            df = df_filt,
                            feature = feature,
                            rows = 10,
                            cols = 10,
                            channel = channel_visualization,
                            seed = 42,
                            pyramid_level = pyramid_lvl_visualization)

# Save DF/AD

In [ ]:
save_after_filtering(
    df = df_filt,
    df_raw = df_raw,
    ad_raw = ad_raw
    )